# Part 5.1 (Extended) — HLCA Experiments: Large-Scale Harmonization + QA 🔬

*Last updated:* 2026-01-08

This notebook is an **advanced extension** of `tutorial_harmonization.ipynb`.

Use it when you want:
- a structured pilot → QA → scale-up workflow
- deeper diagnostics on feature retention (union vs intersect)
- a template for large runs you can adapt for publication-grade analyses

> **Prerequisite:** run `tutorial_harmonization.ipynb` first so the concepts and outputs are familiar.


## 1. Before you start (checklist)

### 1.1 What you need

1. A working Python environment with:
   - IDTrack (Python package: `idtrack`)
   - `anndata` (and usually `scanpy`, depending on your downstream analysis)
2. A folder containing HLCA (or HLCA-like) datasets stored as `.h5ad` files.
3. Disk space: both HLCA data and IDTrack caches can be large.

### 1.2 Two environment variables that make your life easier

- `IDTRACK_LOCAL_REPO` — where IDTrack stores caches and graph snapshots (defaults to `./idtrack_cache`).
- `HLCA_BASE_PATH` — where your HLCA `.h5ad` files live.

Example (macOS/Linux):
```bash
export IDTRACK_LOCAL_REPO="$HOME/idtrack_cache"
export HLCA_BASE_PATH="$HOME/data/HLCA"
```

### 1.3 What you will produce

At the end, you will have:
- a repeatable, documented **harmonization run** (parameters + outputs)
- summary tables that tell you how many genes were kept/removed and why
- a clear workflow you can reuse for other large consortia datasets


## 2. Setup (safe to run)

This section does three things:
1. reads your environment variables
2. creates the local repository folder if needed
3. prepares an `idtrack.API` object

> **Tip:** If you are sharing this notebook with a collaborator, tell them to set `IDTRACK_LOCAL_REPO` and
> `HLCA_BASE_PATH` first — it prevents a lot of confusion.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import idtrack

# Where IDTrack caches everything (tables, graphs, YAML configs)
LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', './idtrack_cache')).resolve()
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

# Where your HLCA .h5ad files live (not shipped with IDTrack)
HLCA_BASE_PATH_RAW = os.environ.get('HLCA_BASE_PATH', '').strip()
HLCA_BASE_PATH = Path(HLCA_BASE_PATH_RAW).expanduser().resolve() if HLCA_BASE_PATH_RAW else None

api = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
api.configure_logger()

print('IDTrack local repository:', LOCAL_REPOSITORY)
print('HLCA base path:', HLCA_BASE_PATH)


## 3. Verify HLCA input data

You should have a folder that contains multiple `.h5ad` files.

A common layout is:
```text
HLCA_BASE_PATH/
  dataset_001.h5ad
  dataset_002.h5ad
  ...
```

> If your HLCA layout is nested (subfolders), you can adapt the file search pattern in the next cell.


In [ ]:
if not HLCA_BASE_PATH or not HLCA_BASE_PATH.exists():
    print('HLCA_BASE_PATH is not set or does not exist. Set it and re-run this cell.')
    hlca_files = []
else:
    # Adjust this if your HLCA files are in subfolders (e.g. HLCA_BASE_PATH.rglob('*.h5ad'))
    hlca_files = sorted(HLCA_BASE_PATH.glob('*.h5ad'))
    print('Found .h5ad files:', len(hlca_files))
    print('First 10:', [p.name for p in hlca_files[:10]])


## 4. Choose your snapshot release (reproducibility knob)

IDTrack’s key idea is the **graph snapshot**:
- you pick a “latest included” Ensembl release
- IDTrack builds a graph that ignores anything newer

Why this matters:
- If you re-run your analysis in 6 months, Ensembl may have new releases.
- Without a snapshot boundary, results can drift.

In this notebook we default to **the latest available human release** (convenient), but for publications you may
prefer to pin to a specific number.


In [ ]:
organism, latest_release = api.resolve_organism('human')

# Option A (default): always use latest available release
TARGET_RELEASE = latest_release

# Option B (recommended for papers): pin to an explicit release
# TARGET_RELEASE = 110

organism, latest_release, TARGET_RELEASE


## 5. Ensure the human graph snapshot is available

If you already ran `initialization_graph.ipynb`, this should *load from cache*.

> Building the graph can take time on first run. If you are doing this on a laptop, consider running a **pilot** first
> (few datasets) and letting the graph build run overnight if needed.


In [ ]:
# Build (or load) the graph snapshot. This is the expensive one-time step.
api.build_graph(organism_name=organism, snapshot_release=TARGET_RELEASE, calculate_caches=True)

g = api.track.graph
print('Graph organism:', g.graph.get('organism'))
print('Graph snapshot release:', g.graph.get('ensembl_release'))
print('Graph nodes:', g.number_of_nodes())
print('Graph edges:', g.number_of_edges())


## 6. Build a dataset dictionary (HLCA datasets)

IDTrack’s harmonization helper expects:

```python
data_h5ad_dict = {
    'dataset_alias': '/absolute/path/to/file.h5ad',
    ...
}
```

Two recommendations:
1. Use **short, stable dataset names** (these show up in outputs and logs).
2. Start with a **pilot subset** (2–3 datasets) to make sure everything works.


In [ ]:
def build_dataset_dict_from_files(files: list[Path], max_datasets: int | None = None) -> dict[str, str]:
    selected = files if max_datasets is None else files[:max_datasets]
    return {p.stem: str(p) for p in selected}


data_h5ad_dict_all = build_dataset_dict_from_files(hlca_files)
data_h5ad_dict_pilot = build_dataset_dict_from_files(hlca_files, max_datasets=3)

print('All datasets:', len(data_h5ad_dict_all))
print('Pilot datasets:', len(data_h5ad_dict_pilot))
list(data_h5ad_dict_pilot)[:10]


## 7. Pilot run (strongly recommended)

This is the workflow you want for a large project:

1. Run a **pilot** (few datasets)
2. Inspect diagnostics
3. Decide policies (namespace, union/intersect, what to do with ambiguous genes)
4. Scale up to full HLCA

### 7.1 Choose your final namespace

Common choices:
- **`HGNC Symbol`**: human-readable, but symbols can be ambiguous or change
- **`ensembl_gene`**: stable and precise, but less human-friendly

In many integration pipelines, **Ensembl IDs are safest**. For presentations and interpretation, symbols are convenient.
You can also keep Ensembl IDs for computation and add symbols as an annotation column later.


In [ ]:
try:
    import anndata as ad  # noqa: F401
except ImportError as e:
    raise ImportError(
        "This notebook requires `anndata` for harmonization. Install it with `pip install anndata` (or via conda)."
    ) from e

# Pick an output folder for this experiment (kept separate from the global IDTrack cache)
project_out = (LOCAL_REPOSITORY / 'hlca_experiments').resolve()
project_out.mkdir(parents=True, exist_ok=True)

# Choose your conversion target
FINAL_DATABASE = 'HGNC Symbol'  # try also: 'ensembl_gene'

harmonizer_pilot = idtrack.HarmonizeFeatures(
    project_name='hlca_pilot',
    data_h5ad_dict=data_h5ad_dict_pilot,
    project_local_repository=str(project_out),
    idtrack_local_repository=str(LOCAL_REPOSITORY),
    target_ensembl_release=TARGET_RELEASE,
    final_database=FINAL_DATABASE,
    organism_name=organism,
    graph_last_ensembl_release=TARGET_RELEASE,
    verbose_level=2,
)

harmonizer_pilot


## 8. Read the diagnostics (this is the most important part)

IDTrack is designed to be explicit about uncertainty. After a pilot run, you should always look at:

1. **Conversion failures (1→0)**
   - identifiers that could not be mapped to your chosen namespace
2. **Ambiguity (1→n)**
   - identifiers that map to multiple targets and require a policy decision
3. **Inconsistency across datasets**
   - the same input label behaving differently in different datasets (often because the label is too ambiguous)

The next cell prints compact summaries.
> Don’t worry if you see failures — in real datasets, some amount of messiness is normal.


In [ ]:
print('Datasets in pilot:', len(harmonizer_pilot.data_h5ad_dict))

print('\n--- Conversion failures (1→0) ---')
print('Count:', len(harmonizer_pilot.conversion_failed_identifiers))
print('Example (first 25):', sorted(list(harmonizer_pilot.conversion_failed_identifiers))[:25])

print('\n--- Failed but consistent (kept) ---')
print('Count:', len(harmonizer_pilot.conversion_failed_but_consistent_identifiers))
print('Example (first 25):', sorted(list(harmonizer_pilot.conversion_failed_but_consistent_identifiers))[:25])

print('\n--- Collapsed ambiguous mappings (1→n; multiple Ensembl candidates) ---')
print('Count:', len(harmonizer_pilot.multiple_ensembl_dict))
some_keys = list(harmonizer_pilot.multiple_ensembl_dict)[:10]
print('Example keys:', some_keys)


## 9. “Union vs Intersect” experiment (feature retention)

When you merge datasets, you must decide what to do with genes that are present in some studies but not others.

- **Union** keeps all genes seen anywhere (missing genes are filled with zeros).
- **Intersect** keeps only genes present everywhere.

At HLCA scale:
- `union` keeps more biology but can create a very wide matrix.
- `intersect` is smaller and faster but can discard study-specific signal.

The next cell performs both merges for the pilot subset.


In [ ]:
# Pilot merge (these can be large; start small)
unified_union = harmonizer_pilot.unify_multiple_anndatas(mode='union')
unified_intersect = harmonizer_pilot.unify_multiple_anndatas(mode='intersect')

print('Union shape (cells x genes):', unified_union.shape)
print('Intersect shape (cells x genes):', unified_intersect.shape)


## 10. Scale up to full HLCA (template)

Once the pilot works, scaling is “just” more data.

### 10.1 Practical advice

1. Increase datasets gradually: 3 → 10 → 30 → all.
2. Use a machine with enough RAM; `.h5ad` loading dominates memory.
3. Keep outputs per run (don’t overwrite) — treat them as provenance.

### 10.2 Full run code (commented out)

Uncomment and run when ready.


In [ ]:
# if data_h5ad_dict_all:
#     project_out_full = (LOCAL_REPOSITORY / 'hlca_full_run').resolve()
#     project_out_full.mkdir(parents=True, exist_ok=True)
#
#     harmonizer_full = idtrack.HarmonizeFeatures(
#         project_name='hlca_full',
#         data_h5ad_dict=data_h5ad_dict_all,
#         project_local_repository=str(project_out_full),
#         idtrack_local_repository=str(LOCAL_REPOSITORY),
#         target_ensembl_release=TARGET_RELEASE,
#         final_database=FINAL_DATABASE,
#         organism_name=organism,
#         graph_last_ensembl_release=TARGET_RELEASE,
#         verbose_level=2,
#     )
#
#     unified_hlca_union = harmonizer_full.unify_multiple_anndatas(mode='union')
#     unified_hlca_union.write_h5ad(project_out_full / 'hlca_union_harmonized.h5ad')
#
#     # Optional: also keep an intersect version
#     # unified_hlca_intersect = harmonizer_full.unify_multiple_anndatas(mode='intersect')
#     # unified_hlca_intersect.write_h5ad(project_out_full / 'hlca_intersect_harmonized.h5ad')


## 11. Interpretation & caveats (please read)

### 11.1 Don’t treat symbols as ground truth

Gene symbols are extremely useful for humans, but they are not perfect identifiers.
If your analysis needs strict reproducibility and precision, consider using `ensembl_gene` as your primary
feature index and storing symbols as an annotation column.

### 11.2 Ambiguity is not an “error”

If IDTrack reports 1→n outcomes, it is telling you that:
- the label is not specific enough, or
- biology/curation changed across releases, or
- the external database is promiscuous

Your job is to decide a policy:
- keep best only
- keep all candidates (rarely good for downstream matrices)
- drop ambiguous genes

### 11.3 Keep the artifacts

For HLCA-scale runs, treat the following as part of your analysis provenance:
- the snapshot release number
- the external YAML used to build the graph
- the output folder produced by harmonization


## 12. Summary and next steps

You now have a practical, scalable workflow for HLCA-like harmonization:
1. build/load a reproducible **human graph snapshot**
2. run a **pilot** harmonization
3. inspect diagnostics and decide policies
4. scale to the full dataset set

Next recommended tutorial:
- `tutorial_humanization_mouse_pig_to_human.ipynb` (cross-species mapping; advanced)
